# AVGO Stock Direction Prediction — 3 Models

**Dataset:** Broadcom (AVGO) 1-minute OHLCV data
**Target:** Predict whether next-minute close price will go UP (1) or DOWN (0)

**Models:**
1. K-Nearest Neighbors (KNN)
2. Gaussian Naive Bayes
3. Support Vector Machine (SVM)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

sns.set_style('darkgrid')
print('Libraries loaded.')


## 1. Load & Prepare Data

We concatenate train/val/test chronologically to build the target variable (next-period direction), then re-split.


In [ ]:
# Load
train = pd.read_csv('avgo_train.csv', index_col=0)
val = pd.read_csv('avgo_val.csv', index_col=0)
test = pd.read_csv('avgo_test.csv', index_col=0)

print(f'Train: {train.shape}, Val: {val.shape}, Test: {test.shape}')

# Concatenate chronologically
df = pd.concat([train, val, test])
print(f'Combined: {df.shape}')

# Build target: 1 if next close > current close, else 0
df['target'] = (df['close'].shift(-1) > df['close']).astype(int)

# Drop last row (no next close)
df = df.dropna(subset=['target'])
df['target'] = df['target'].astype(int)
print(f'After target creation: {df.shape}')
print(f'Target distribution:\n{df["target"].value_counts(normalize=True).mul(100).round(1)}')


## 2. Feature Selection

Features: all numeric columns except close/open/high/low/volume (raw levels) and target. We keep engineered features only.


In [ ]:
# Define features — exclude raw price/volume levels, keep engineered
exclude = ['open', 'high', 'low', 'close', 'volume', 'target']
feature_cols = [c for c in df.columns if c not in exclude]
print(f'Feature count: {len(feature_cols)}')
print(f'Features: {feature_cols}')

X = df[feature_cols].values
y = df['target'].values

# Re-split (same proportions)
n = len(df)
train_end = int(n * 0.7)
val_end = int(n * 0.85)

X_train, y_train = X[:train_end], y[:train_end]
X_val, y_val = X[train_end:val_end], y[train_end:val_end]
X_test, y_test = X[val_end:], y[val_end:]

print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')
print(f'Train target distribution: {np.mean(y_train)*100:.1f}% UP')


## 3. Model Training & Evaluation


In [ ]:
results = []

def evaluate_model(name, model, X_tr, y_tr, X_va, y_va, X_te, y_te):
    model.fit(X_tr, y_tr)
    
    train_acc = accuracy_score(y_tr, model.predict(X_tr))
    val_acc = accuracy_score(y_va, model.predict(X_va))
    test_acc = accuracy_score(y_te, model.predict(X_te))
    test_f1 = f1_score(y_te, model.predict(X_te))
    test_precision = precision_score(y_te, model.predict(X_te))
    test_recall = recall_score(y_te, model.predict(X_te))
    
    results.append({
        'Model': name,
        'Train Acc': f'{train_acc:.4f}',
        'Val Acc': f'{val_acc:.4f}',
        'Test Acc': f'{test_acc:.4f}',
        'Test F1': f'{test_f1:.4f}',
        'Test Precision': f'{test_precision:.4f}',
        'Test Recall': f'{test_recall:.4f}'
    })
    
    print(f'\n=== {name} ===')
    print(f'Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | Test Acc: {test_acc:.4f}')
    print(f'Test F1: {test_f1:.4f} | Precision: {test_precision:.4f} | Recall: {test_recall:.4f}')
    print(f'\nClassification Report:\n{classification_report(y_te, model.predict(X_te))}')
    
    # Confusion Matrix
    cm = confusion_matrix(y_te, model.predict(X_te))
    plt.figure(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['DOWN','UP'], yticklabels=['DOWN','UP'])
    plt.title(f'{name} — Confusion Matrix')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.tight_layout()
    plt.savefig(f'cm_{name.lower().replace(" ", "_")}.png', dpi=100)
    plt.show()
    print(f'[Saved: cm_{name.lower().replace(" ", "_")}.png]')


### 3.1 K-Nearest Neighbors (KNN)

Tuning `n_neighbors` on validation set.


In [ ]:
# Tuning KNN
best_k = 1
best_val = 0
for k in [3, 5, 7, 9, 11, 15, 21, 31]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    val_acc = accuracy_score(y_val, knn.predict(X_val))
    print(f'  k={k:2d} → Val Acc: {val_acc:.4f}')
    if val_acc > best_val:
        best_val = val_acc
        best_k = k

print(f'\nBest k = {best_k} (Val Acc: {best_val:.4f})')

knn = KNeighborsClassifier(n_neighbors=best_k)
evaluate_model('KNN (k=' + str(best_k) + ')', knn, X_train, y_train, X_val, y_val, X_test, y_test)


### 3.2 Gaussian Naive Bayes


In [ ]:
gnb = GaussianNB()
evaluate_model('Gaussian Naive Bayes', gnb, X_train, y_train, X_val, y_val, X_test, y_test)


### 3.3 Support Vector Machine (SVM)

Tuning kernel and C on validation set.


In [ ]:
# Tuning SVM
best_svm = None
best_svm_name = ''
best_svm_val = 0

for kernel in ['linear', 'rbf']:
    for C in [0.1, 1, 10]:
        svm = SVC(kernel=kernel, C=C, random_state=42)
        svm.fit(X_train, y_train)
        val_acc = accuracy_score(y_val, svm.predict(X_val))
        print(f'  kernel={kernel:6s} C={C:4} → Val Acc: {val_acc:.4f}')
        if val_acc > best_svm_val:
            best_svm_val = val_acc
            best_svm = svm
            best_svm_name = f'SVM ({kernel}, C={C})'

print(f'\nBest: {best_svm_name} (Val Acc: {best_svm_val:.4f})')

evaluate_model(best_svm_name, best_svm, X_train, y_train, X_val, y_val, X_test, y_test)


## 4. Model Comparison


In [ ]:
# Comparison table
results_df = pd.DataFrame(results)
print('\n' + '='*80)
print('MODEL COMPARISON')
print('='*80)
print(results_df.to_string(index=False))
print('='*80)

# Visual comparison
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

metrics = ['Test Acc', 'Test F1', 'Test Precision']
colors = ['#2ecc71', '#3498db', '#e74c3c']

for i, metric in enumerate(metrics):
    ax = axes[i]
    vals = [float(r[metric]) for r in results]
    bars = ax.bar([r['Model'] for r in results], vals, color=colors)
    ax.set_title(metric, fontsize=12, fontweight='bold')
    ax.set_ylim(0, 1)
    ax.tick_params(axis='x', rotation=15)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
                f'{v:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=120)
plt.show()
print('[Saved: model_comparison.png]')


## 5. Summary & Insights


In [ ]:
print('='*60)
print('FINAL SUMMARY — AVGO Stock Direction Prediction')
print('='*60)
print(f'Total samples: {len(df)}')
print(f'Features: {len(feature_cols)}')
print(f'Class balance: {df["target"].value_counts().to_dict()}')
print()
print('Results:')
for r in results:
    print(f'  {r["Model"]:30s} | Test Acc: {r["Test Acc"]} | F1: {r["Test F1"]}')
print()
best = max(results, key=lambda r: float(r['Test F1']))
print(f'🏆 Best model: {best["Model"]} (F1={best["Test F1"]})')
print('='*60)
